# 實驗：WGAN-GP 人臉生成

- 本章主題是 WGAN-GP（Wasserstein GAN with Gradient Penalty），目標是訓練能穩定生成人臉影像的生成模型。
- 透過 Wasserstein 距離與 Gradient Penalty（梯度懲罰）來提升訓練穩定性，減少傳統 GAN 常見的模式崩潰問題。
- 資料集使用 Kaggle 的 UTKFace Cropped Dataset: https://www.kaggle.com/datasets/moritzm00/utkface-cropped

In [ ]:
# 解壓縮 UTKFace 資料集到訓練資料夾
# 如果上傳的 zip 檔名不同，請同步修改這裡的檔名
!unzip utk_face.zip -d images_folder/

## 1. Import套件

In [ ]:
!pip install tensorboard

import os
import re

import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.optim as optim
import torchvision.transforms as transforms
from torch import nn
from torch.autograd import grad
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision.utils import make_grid


## 2. 網路模型定義


In [ ]:
# 定義動態計算區塊
# 這個 block 會把輸入分成多條分支，逐步產生不同深度的特徵，
# 最後再 concat 起來融合，並可選擇加入 residual connection
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, use_residual=True, groups=2):
        super(ConvBlock, self).__init__()
        self.use_residual = use_residual

        # 第一條 1x1 分支將通道數壓到 out_c / 2
        self.p1 = nn.Sequential(
            nn.Conv2d(in_c, out_c // 2, 1, 1, groups=groups),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # 第二條 1x1 分支作為後續 depthwise 與 group convolution 的起點
        self.p2 = nn.Sequential(
            nn.Conv2d(in_c, out_c // 2, 1, 1, groups=groups),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # 先用 depthwise convolution 擷取各通道的空間特徵，
        # 再用 group convolution 混合部分通道資訊
        self.pw = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(out_c // 2, out_c // 2, 3, 1, 1, groups=out_c // 2),
                nn.Conv2d(out_c // 2, out_c // 2, 3, 1, 1, groups=groups),
                nn.LeakyReLU(0.2, inplace=True),
            ) for _ in range(2)
        ])

        # concat 後通道數為 out_c * 2，再壓縮並恢復到 out_c
        self.merge = nn.Sequential(
            nn.Conv2d(out_c * 2, out_c // 2, 1, 1, groups=groups),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_c // 2, out_c, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # residual 的通道數需要和主分支輸出一致才能相加
        if use_residual and in_c != out_c:
            self.residual = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, 1),
                nn.LeakyReLU(0.2, inplace=True),
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x):
        residual = self.residual(x)

        # x1、x2 來自原始輸入，x3、x4 是在 x2 上逐步加深的特徵
        x1 = self.p1(x)
        x2 = self.p2(x)
        x3 = self.pw[0](x2)
        x4 = self.pw[1](x3)

        x = torch.cat([x1, x2, x3, x4], dim=1)
        x = self.merge(x)

        # residual connection 可幫助梯度流動，讓深層網路更容易訓練
        if self.use_residual:
            x = x + residual

        return x


# 定義上採樣區塊
# 使用 ConvTranspose2d 將 feature map 放大 2 倍，再用 convolution 整理特徵
class UpscaleBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super(UpscaleBlock, self).__init__()

        self.upscale = nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c // 2, 4, 2, 1),
            nn.BatchNorm2d(out_c // 2),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.conv = nn.Sequential(
            nn.Conv2d(out_c // 2, out_c // 2, 3, 1, 1),
            nn.BatchNorm2d(out_c // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_c // 2, out_c, 1, 1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        x = self.upscale(x)
        x = self.conv(x)
        return x


class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        self.dim = z_dim

        # 將 latent vector z 映射成 4x4 的初始 feature map
        self.mapping = nn.Sequential(
            nn.Linear(self.dim, self.dim // 2, bias=False),
            nn.Linear(self.dim // 2, self.dim, bias=False),
            nn.Linear(self.dim, self.dim * 4 * 4),
        )

        # Decoder 逐層上採樣，4x4 -> 8x8 -> 16x16 -> 32x32 -> 64x64
        # 同時逐步降低通道數，最後輸出 RGB 圖片
        self.decoder = nn.Sequential(
            nn.Sequential(
                UpscaleBlock(self.dim, 512),
                ConvBlock(512, 512),
            ),
            nn.Sequential(
                UpscaleBlock(512, 256),
                ConvBlock(256, 256),
            ),
            nn.Sequential(
                UpscaleBlock(256, 128),
                ConvBlock(128, 128),
            ),
            nn.Sequential(
                UpscaleBlock(128, 64),
                ConvBlock(64, 64),
            ),
            nn.Sequential(
                # Tanh 讓輸出範圍對齊前處理後的圖片數值
                nn.Conv2d(64, 3, 3, 1, 1),
                nn.Tanh()
            )
        )

    def forward(self, z):
        x = self.mapping(z)
        x = x.view(-1, self.dim, 4, 4)
        x = self.decoder(x)
        return x


# 定義 Discriminator / Critic
# 在 WGAN-GP 中它輸出的是實數分數，不是分類機率
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        # WGAN-GP 不使用 BatchNorm，避免 batch 之間互相影響 gradient penalty
        # 64x64 影像會被連續下採樣到 4x4 特徵圖
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 64, 5, 2, 2, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 5, 2, 2, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 5, 2, 2, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 5, 2, 2, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # 512 * 4 * 4 = 8192，最後輸出單一 critic score
        self.dis_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(8192, 1),
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        dis = self.dis_head(x)
        return dis

## 3. Dataset

In [ ]:
class UTKFaceDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.filenames = []

        # UTKFace 檔名包含年齡與其他標籤，這裡只保留符合命名規則的圖片
        for filename in os.listdir(data_dir):
            m = re.match(r"(\d+)_\d+.*\.(jpg|jpeg|png)$", filename, re.IGNORECASE)
            if m is None:
                continue
            self.filenames.append(filename)

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        image = Image.open(os.path.join(self.data_dir, filename)).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image


# Dataset 設定
DATA_DIR    = '/content/images_folder/UTKFace'
batch_size  = 256
num_workers = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# 將圖片統一成 64x64，並正規化到 Tanh 輸出範圍
transform = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = UTKFaceDataset(data_dir=DATA_DIR, transform=transform)
print(f'Dataset size: {len(dataset)}')

# drop_last 確保每個 batch 大小固定，方便固定雜訊與紀錄對齊
dataloader = DataLoader(
    dataset, batch_size=batch_size, shuffle=True,
    drop_last=True, num_workers=num_workers,
    pin_memory=True
)

## 4. 定義 gradient_penalty 計算函式


In [ ]:
def compute_gradient_penalty(dis, real_samples, fake_samples, device):
    """計算 WGAN-GP 的 gradient penalty。"""
    # 在真實與生成圖片之間插值，檢查 Critic 的梯度是否接近 1
    alpha = torch.rand(real_samples.size(0), 1, 1, 1, device=device)
    interpolated_images = (alpha * real_samples + (1 - alpha) * fake_samples.detach()).requires_grad_(True)
    d_interpolated = dis(interpolated_images)

    # 對插值圖片反向求梯度，作為 Lipschitz constraint 的懲罰依據
    gradients = grad(
        outputs=d_interpolated,
        inputs=interpolated_images,
        grad_outputs=torch.ones_like(d_interpolated),
        create_graph=True,
        retain_graph=True,
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

## 5. 訓練迴圈


In [ ]:
RUNS_DIR   = '/content/runs'
lr         = 3e-4
z_dim      = 256
num_epochs = 50
lambda_gp  = 10

# TensorBoard 會記錄訓練曲線、生成圖片與權重分布
writer = SummaryWriter(RUNS_DIR)

# 初始化 Generator 和 Critic
generator = Generator(z_dim=z_dim).to(device)
discriminator = Discriminator().to(device)

# WGAN-GP 建議使用 Adam，並將 beta1 設為 0 以提升訓練穩定性
g_optimizer = optim.Adam(generator.parameters(), lr=lr, betas=(0.0, 0.9))
d_optimizer = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.0, 0.9))

# 固定同一組雜訊，方便觀察每個 epoch 的生成品質變化
fixed_noise = torch.randn(batch_size, z_dim, device=device)
post_str = ''

for epoch in range(num_epochs):
    tqdm_loader = tqdm(dataloader, desc=f'Epoch {epoch + 1}/{num_epochs}', dynamic_ncols=True)
    if post_str != '':
        tqdm_loader.set_postfix_str(post_str)

    d_loss, g_loss, real_img = None, None, None

    for i, real_images in enumerate(tqdm_loader):
        real_images = real_images.to(device)
        real_img = real_images

        # Critic 更新次數比 Generator 多，讓分數估計先保持穩定
        if i % 3 < 2:
            z = torch.randn(real_images.size(0), z_dim, device=device)
            fake_images = generator(z)

            # WGAN loss 希望真圖分數高、假圖分數低，並加入 gradient penalty
            d_real = discriminator(real_images)
            d_fake = discriminator(fake_images.detach())
            gradient_penalty = lambda_gp * compute_gradient_penalty(discriminator, real_images, fake_images, device)

            d_loss = -d_real.mean() + d_fake.mean() + gradient_penalty

            discriminator.zero_grad()
            d_loss.backward()
            d_optimizer.step()
        else:
            z = torch.randn(real_images.size(0), z_dim, device=device)
            fake_images = generator(z)
            d_fake = discriminator(fake_images)

            # Generator 目標是讓假圖拿到更高的 critic score
            g_loss = -d_fake.mean()

            generator.zero_grad()
            g_loss.backward()
            g_optimizer.step()

    if d_loss is not None and g_loss is not None:
        post_str = f'Epoch [{epoch + 1}/{num_epochs}], D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}'

    with torch.no_grad():
        if epoch == 0:
            try:
                writer.add_graph(generator, torch.randn(batch_size, z_dim, device=device))
            except Exception as e:
                print('add_graph skipped:', e)

        if real_img is not None:
            writer.add_images('real', (real_img + 1) / 2, global_step=epoch)
            writer.add_histogram('real_hist', real_img, global_step=epoch)

        if d_loss is not None:
            writer.add_scalar('loss/dis', d_loss.item(), epoch)
        if g_loss is not None:
            writer.add_scalar('loss/gen', g_loss.item(), epoch)

        # 權重分布可協助觀察訓練是否出現爆炸或塌陷
        for name, v in generator.named_parameters():
            writer.add_histogram(name, v, global_step=epoch)

        sample_images = generator(fixed_noise)
        fake_grid = make_grid((sample_images + 1) / 2, nrow=8)
        writer.add_image('output/faces', fake_grid, global_step=epoch)

writer.close()


## 6. 使用Generator生成圖片

In [ ]:
generator.eval()

# 使用訓練好的 Generator 產生一組新臉部圖片
n_samples = 16
with torch.no_grad():
    z = torch.randn(n_samples, z_dim, device=device)
    imgs = generator(z)
    grid = make_grid((imgs + 1) / 2, nrow=8).cpu().permute(1, 2, 0).numpy()
    plt.figure(figsize=(12, 6))
    plt.title('Generated faces')
    plt.axis('off')
    plt.imshow(grid)
    plt.show()

## 7. 啟動 TensorBoard

In [ ]:
# 開啟 TensorBoard，查看訓練過程中的 loss、圖片與權重分布
%load_ext tensorboard
%tensorboard --logdir runs